<a href="https://colab.research.google.com/github/eric20041027/Data_Mining/blob/main/notebooks/train_pubmedbert_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# === 0. 環境設定：clone repo、裝套件、掛 Drive、確認 GPU ===
import os, sys, shutil

REPO_DIR = '/content/Data_Mining'
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/eric20041027/Data_Mining.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

# 掛 Drive
from google.colab import drive
drive.mount('/content/drive')

# GPU 確認
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('bf16 supported:', torch.cuda.is_bf16_supported())

# 安裝套件
!pip install -q -U "transformers>=4.44,<4.50" "accelerate>=0.33" "datasets>=2.20" "scikit-learn>=1.4"
print('\nSetup OK.')

In [ ]:
# === 1. 從 Drive 還原舊的 prediction artifacts (val_probs/test_probs/args)===
import tarfile, os

candidates = [
    '/content/drive/MyDrive/Kaggle_backup/predictions_only_after_noweight.tar.gz',
    '/content/drive/MyDrive/Kaggle_backup/predictions_only_before_noweight.tar.gz',
    '/content/drive/MyDrive/Kaggle_backup/predictions_only.tar.gz',
]
for tar_path in candidates:
    if os.path.exists(tar_path):
        print(f'Restoring from: {tar_path}')
        with tarfile.open(tar_path) as tar:
            tar.extractall('/content/Data_Mining')
        print('  done')
        break

# 檢查還原狀態
src = '/content/Data_Mining/outputs/bert_runs'
if os.path.isdir(src):
    dirs = sorted(d for d in os.listdir(src) if os.path.isdir(os.path.join(src, d)))
    print(f'\n=== 目前 {len(dirs)} 個 run 資料夾 ===')
    for d in dirs:
        print(f'  {d}')
else:
    print('outputs/bert_runs 不存在，將從零開始訓練')

In [ ]:
# === 2. 共用工具：訓練、備份、ensemble ===
import os, subprocess, tarfile, glob, json
import pandas as pd

def train_model(model, seed, epochs, bs, lr, tag_prefix, class_weight='none'):
    """訓練單一模型 5-fold"""
    for fold in range(5):
        cmd = (
            f'python src/train_bert.py --model {model} '
            f'--fold {fold} --seed {seed} --epochs {epochs} --batch-size {bs} '
            f'--lr {lr} --max-length 512 --class-weight {class_weight} '
            f'--tag {tag_prefix}_fold{fold}'
        )
        print('>>>', cmd)
        rc = os.system(cmd)
        assert rc == 0, f'{tag_prefix} fold {fold} failed'

    # 印 OOF 摘要
    runs = sorted(glob.glob(f'outputs/bert_runs/{tag_prefix}_fold*/metrics.json'))
    df = pd.DataFrame([json.load(open(p)) for p in runs])
    print(f'\n{tag_prefix} per-fold val Macro F1:')
    print(df[['fold', 'val_macro_f1', 'train_secs']])
    print(f'{tag_prefix} mean OOF Macro F1: {df.val_macro_f1.mean():.4f}\n')

def backup(label):
    """輕量備份 (~50MB)：只打包 .npy/.json/.csv"""
    src = '/content/Data_Mining/outputs/bert_runs'
    dst = f'/content/drive/MyDrive/Kaggle_backup/predictions_only_{label}.tar.gz'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    files_to_pack = []
    for run in sorted(os.listdir(src)):
        rd = os.path.join(src, run)
        if not os.path.isdir(rd):
            continue
        for pat in ['*.npy', '*.json', '*.csv']:
            files_to_pack += glob.glob(os.path.join(rd, pat))
    with tarfile.open(dst, 'w:gz') as tar:
        for f in files_to_pack:
            tar.add(f, arcname=os.path.relpath(f, '/content/Data_Mining'))
    size = subprocess.check_output(['du', '-h', dst]).decode().split()[0]
    print(f'Backup: {dst} ({size})')

def run_ensemble(patterns, tag):
    """跑 ensemble 並印關鍵指標"""
    cmd = ['python', 'src/ensemble_predict.py'] + ['--bert-runs'] + patterns + ['--no-overlap-constraint', '--tag', tag]
    print('>>>', ' '.join(cmd))
    out = subprocess.run(cmd, cwd='/content/Data_Mining', capture_output=True, text=True)
    if out.returncode != 0:
        print('STDERR:', out.stderr)
        return
    lines = out.stdout.split('\n')
    for line in lines:
        if any(k in line for k in ['Found ', 'fold ', 'OOF Macro F1', 'macro avg', 'general path']):
            print(line)
    print()

print('工具函數已載入。')

In [ ]:
# === 3. BioBERT noweight 5-fold (~50 分鐘) ===
train_model(
    model='dmis-lab/biobert-base-cased-v1.2',
    seed=42, epochs=4, bs=32, lr=2e-5,
    tag_prefix='biobert_noweight_seed42',
    class_weight='none'
)
backup('after_biobert_noweight')

In [ ]:
# === 4. PubMedBERT noweight seed=2024 5-fold (~50 分鐘) ===
train_model(
    model='microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
    seed=2024, epochs=4, bs=32, lr=2e-5,
    tag_prefix='pubmedbert_noweight_seed2024',
    class_weight='none'
)
backup('after_pubmedbert_noweight_2024')

In [ ]:
# === 5. 中途 ensemble，看 3 個 noweight 模型合起來表現 ===
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
], 'v12_3noweight')

In [ ]:
# === 6. PubMedBERT-large noweight (~80 分鐘) ===
train_model(
    model='microsoft/BiomedNLP-BiomedBERT-large-uncased-abstract',
    seed=42, epochs=3, bs=16, lr=1e-5,
    tag_prefix='pubmedbertlarge_noweight_seed42',
    class_weight='none'
)
backup('final_all_noweight')

In [ ]:
# === 7. 全部 noweight 模型 ensemble，產生最終 submissions ===

# 候選 A: 純 noweight (seed=42)
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed42_fold*',
], 'final_a_noweight42')

# 候選 B: 2 個 PubMedBERT noweight seeds
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
], 'final_b_pubmed_2seeds')

# 候選 C: PubMedBERT × 2 + BioBERT (3 個 base noweight)
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
], 'final_c_3noweight')

# 候選 D: 全部 4 個 noweight 模型（含 large）
run_ensemble([
    'outputs/bert_runs/pubmedbert_noweight_seed*_fold*',
    'outputs/bert_runs/biobert_noweight_seed*_fold*',
    'outputs/bert_runs/pubmedbertlarge_noweight_seed*_fold*',
], 'final_d_4noweight')

# 顯示所有 final submission
print('\n=== 所有 final submissions ===')
import os, hashlib, pandas as pd
for tag in ['final_a_noweight42', 'final_b_pubmed_2seeds', 'final_c_3noweight', 'final_d_4noweight']:
    p = f'outputs/submission_{tag}.csv'
    if os.path.exists(p):
        h = hashlib.md5(open(p, 'rb').read()).hexdigest()[:10]
        df = pd.read_csv(p)
        dist = df['label'].value_counts(normalize=True).sort_index().round(3).to_dict()
        print(f'  {tag}: md5={h} dist={dist}')

# 下載
from google.colab import files
for tag in ['final_a_noweight42', 'final_b_pubmed_2seeds', 'final_c_3noweight', 'final_d_4noweight']:
    p = f'/content/Data_Mining/outputs/submission_{tag}.csv'
    if os.path.exists(p):
        files.download(p)

print('\n=== 全部完成 ===')